In [ ]:
import pandas as pd

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


In [ ]:


df = pd.read_csv("/kaggle/input/datasets/joelleiliovits/new-data-csv/new_data.csv")
df = df.dropna(subset=["text", "tags"]).reset_index(drop=True)

df["num_tags"] = df["tags"].apply(lambda x: len(str(x).split()))

# نسب الزيادة
MULTIPLIER = {3: 2, 4: 5, 5: 10}

df = df[df["num_tags"].isin(MULTIPLIER.keys())].copy()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

model = model.to(device)

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

In [ ]:
def paraphrase_batch(texts, batch_size=32):
    results = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Generating"):
        batch_texts = texts[i:i + batch_size]
        prompts = [
            "Paraphrase this programming problem but keep all technical terms:\n" + str(t)
            for t in batch_texts
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.module.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=1)

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        results.extend(decoded)

    return results


In [ ]:
from tqdm.auto import tqdm
aug_texts = []
aug_tags = []
aug_num_tags = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Preparing rows"):
    n = row["num_tags"]

    if n not in MULTIPLIER:
        continue

    repeat = MULTIPLIER[n]

    for _ in range(repeat):
        aug_texts.append(row["text"])
        aug_tags.append(row["tags"])
        aug_num_tags.append(n)

generated_texts = paraphrase_batch(aug_texts, batch_size=32)
df_aug = pd.DataFrame({
    "text": generated_texts,
    "Tags": aug_tags
})

df_aug.to_csv("/kaggle/working/t5_augmented.csv", index=False)

In [ ]:
import torch

print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("arch list:", torch.cuda.get_arch_list())
print("gpu count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))